# KADMON — Exporter les centroïdes QuickBundles en NumPy

Ce notebook compresse des bundles avec QuickBundles, puis exporte **uniquement leurs centroïdes** dans des fichiers `.npy`. Les poids des clusters ne sont pas exportés. Deux modes sont disponibles : `pair` pour une source et une cible, ou `all` pour parcourir récursivement tout le dossier de bundles.

In [ ]:
from pathlib import Path
import sys

# Fonctionne lorsque Jupyter est lancé depuis la racine ou depuis notebooks/.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "kadmon").is_dir() else CURRENT_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

BUNDLES_DIR = PROJECT_ROOT / "notebooks" / "bundles"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "centroids_export"

# "pair" : exporter SOURCE_PATH et TARGET_PATH seulement.
# "all"  : exporter tous les fichiers correspondant à INPUT_GLOB.
MODE = "pair"
INPUT_GLOB = "**/*.trk"  # Utiliser "**/*.npy" pour les caches NumPy.

SOURCE_PATH = BUNDLES_DIR / "103818/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m.trk"
TARGET_PATH = BUNDLES_DIR / "433839/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m.trk"

N_POINTS = 12
QUICKBUNDLES_THRESHOLD_MM = 7.0

## Sélectionner les bundles

In [ ]:
if MODE == "pair":
    export_jobs = [
        (SOURCE_PATH, OUTPUT_DIR / "source_centroids.npy"),
        (TARGET_PATH, OUTPUT_DIR / "target_centroids.npy"),
    ]
elif MODE == "all":
    input_paths = sorted(path for path in BUNDLES_DIR.glob(INPUT_GLOB) if path.is_file())
    export_jobs = []
    for input_path in input_paths:
        relative = input_path.relative_to(BUNDLES_DIR)
        output_name = f"{relative.stem}_centroids.npy"
        export_jobs.append((input_path, OUTPUT_DIR / relative.parent / output_name))
else:
    raise ValueError("MODE doit être 'pair' ou 'all'.")

if not export_jobs:
    raise FileNotFoundError(f"Aucun bundle trouvé dans {BUNDLES_DIR} avec {INPUT_GLOB!r}.")

print(f"{len(export_jobs)} bundle(s) à exporter :")
for input_path, output_path in export_jobs:
    print(f"- {input_path.relative_to(BUNDLES_DIR)} → {output_path.relative_to(OUTPUT_DIR)}")

## Exporter uniquement les centroïdes

In [ ]:
import numpy as np
from kadmon.compression import compress_quickbundles
from kadmon.io import load_bundle

export_results = []
for input_path, output_path in export_jobs:
    if not input_path.is_file():
        raise FileNotFoundError(f"Bundle introuvable : {input_path}")

    # Traitement séquentiel : un seul bundle et ses centroïdes en mémoire à la fois.
    bundle = load_bundle(input_path, n_points=N_POINTS)
    centroids, _ = compress_quickbundles(
        bundle, threshold=QUICKBUNDLES_THRESHOLD_MM
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(output_path, centroids)
    export_results.append((input_path, output_path, len(bundle), centroids.shape))
    print(f"{input_path.name}: {len(bundle)} streamlines → {len(centroids)} centroïdes")

print(f"Export terminé : {len(export_results)} fichier(s) dans {OUTPUT_DIR}")

## Vérifier les fichiers exportés

In [ ]:
for _, output_path, _, expected_shape in export_results:
    saved = np.load(output_path, mmap_mode="r")
    assert saved.shape == expected_shape
    assert saved.ndim == 3 and saved.shape[1:] == (N_POINTS, 3)

print(f"Export vérifié : {len(export_results)} fichier(s) NumPy valides.")